In [1]:
import os
import glob
from datetime import datetime
import pandas as pd
from IPython.display import display
import re # Import the regular expression module

# --- Mount Google Drive ---
from google.colab import drive
drive.mount('/content/drive')

# =========================
# Configuration (edit if needed or use form fields below)
# =========================
SEARCH_DIR = "/content/to_merge" # @param {"type":"string"}
INCLUDE_SUBFOLDERS = True # @param {type:"boolean"} # Search subfolders under SEARCH_DIR
FILE_PATTERNS_STR = "*.csv,*.xlsx,*.xls" # @param {type:"string"} # Comma-separated file patterns (e.g., "*.csv,*.xlsx")
FILE_PATTERNS = [p.strip() for p in FILE_PATTERNS_STR.split(',')]
MIN_REQUIRED_FILES = 2 # @param {type:"integer"} # Require at least this many files (any mix)

STOP_ON_SANITY_MISMATCH = True # @param {type:"boolean"} # If True, raise an error if row counts don't add up
OUTPUT_DRIVE_DIR = '/content/drive/MyDrive/merged' # @param {"type":"string"}
OUTPUT_BASENAME = 'merged_2025_herewego' # @param {"type":"string"}

# Choose output format: 'csv' or 'xlsx'
OUTPUT_FORMAT = 'xlsx'    # @param ["csv", "xlsx"] {type:"string"}
# =========================

def find_files(base_dir, patterns, recursive=True):
    files = []
    for pat in patterns:
        pattern = os.path.join(base_dir, '**', pat) if recursive else os.path.join(base_dir, pat)
        files.extend(glob.glob(pattern, recursive=recursive))
    files = [f for f in sorted(set(files)) if os.path.isfile(f) and not os.path.basename(f).startswith('.')]
    return files

def read_csv_flex(path):
    """
    Robust CSV reader:
    - First try delimiter inference (sep=None with python engine).
    - If that fails, try common delimiters and encodings.
    """
    try:
        return pd.read_csv(path, sep=None, engine='python')
    except Exception as e_first:
        # try common delimiters
        for sep in [',', ';', '\t', '|']:
            try:
                return pd.read_csv(path, sep=sep, engine='python')
            except Exception:
                pass
        # try common encodings
        for enc in ['utf-8', 'utf-8-sig', 'latin-1']:
            try:
                return pd.read_csv(path, sep=None, engine='python', encoding=enc)
            except Exception:
                pass
        raise RuntimeError(f"Could not read '{path}'. Last error: {e_first}")

def read_excel_flex(path):
    """
    Excel reader that:
    - Opens the workbook
    - Picks the first non-empty sheet (has at least one column and >=1 row)
    - Returns a DataFrame
    """
    try:
        # ExcelFile lets us inspect sheet names cheaply
        xls = pd.ExcelFile(path)  # engine auto-detected (openpyxl on .xlsx)
    except Exception as e:
        # Fallback: try specifying engine explicitly
        try:
            xls = pd.ExcelFile(path, engine='openpyxl')
        except Exception as e2:
            raise RuntimeError(f"Could not open Excel file '{path}'. Errors: {e} / {e2}")

    chosen_sheet = None
    for sheet in xls.sheet_names:
        try:
            # Read a small sample to test if it's non-empty
            df_head = pd.read_excel(xls, sheet_name=sheet, nrows=5)
            if df_head.shape[1] > 0 and df_head.shape[0] > 0:
                chosen_sheet = sheet
                break
        except Exception:
            continue

    if chosen_sheet is None:
        # If all small reads failed or all sheets appear empty, try the first sheet fully
        chosen_sheet = xls.sheet_names[0]

    try:
        df_full = pd.read_excel(xls, sheet_name=chosen_sheet)
    except Exception as e_read:
        # Final fallback with explicit engine
        df_full = pd.read_excel(path, sheet_name=chosen_sheet, engine='openpyxl')

    return df_full

def read_any_flex(path):
    ext = os.path.splitext(path.lower())[1]
    if ext in ['.csv']:
        return read_csv_flex(path)
    elif ext in ['.xlsx', '.xls']:
        return read_excel_flex(path)
    else:
        raise ValueError(f"Unsupported file type for '{path}'")

# --- Discover files ---
paths = find_files(SEARCH_DIR, FILE_PATTERNS, recursive=INCLUDE_SUBFOLDERS)
if len(paths) < MIN_REQUIRED_FILES:
    raise ValueError(
        f"Found {len(paths)} file(s) under '{SEARCH_DIR}'. "
        f"Please upload at least {MIN_REQUIRED_FILES} files (CSV/XLSX/XLS) and re-run."
    )

# --- Read and align schemas (same header names; order follows the first file) ---
dfs = []
row_counts = []
schemas = []

first_df = read_any_flex(paths[0])
# Drop any 'Unnamed:' columns from the first dataframe
first_df = first_df.loc[:, ~first_df.columns.str.contains('^Unnamed')]

if first_df.empty:
    raise AssertionError(f"First file appears empty: {os.path.basename(paths[0])}")
ref_cols = list(first_df.columns)
dfs.append(first_df[ref_cols])
row_counts.append(len(first_df))
schemas.append((paths[0], first_df.shape, ref_cols))

for p in paths[1:]:
    df = read_any_flex(p)
    # Drop any 'Unnamed:' columns from subsequent dataframes
    df = df.loc[:, ~df.columns.str.contains('^Unnamed')]

    if df.empty:
        raise AssertionError(f"File appears empty: {os.path.basename(p)}")

    # Ensure same column *names* (order can differ); then reorder to match first file
    if set(df.columns) != set(ref_cols):
        missing = [c for c in ref_cols if c not in df.columns]
        extra = [c for c in df.columns if c not in ref_cols]
        raise AssertionError(
            f"Schema mismatch in {os.path.basename(p)}.\n"
            f"Missing columns: {missing}\nExtra columns: {extra}\n"
            "All files must have the same header names."
        )
    df = df.reindex(columns=ref_cols)
    dfs.append(df)
    row_counts.append(len(df))
    schemas.append((p, df.shape, list(df.columns)))

# --- Append ---
merged = pd.concat(dfs, ignore_index=True)

# --- Sanity check: row counts must add up ---
sum_input_rows = sum(row_counts)
merged_rows = len(merged)
equation = " + ".join(str(n) for n in row_counts)

print('--- Merge Summary ---')
print(f'Files merged: {len(paths)}')
for (p, shape, cols) in schemas:
    print(f'- {os.path.basename(p)}: {shape[0]} rows, {shape[1]} cols')

print('\n--- Sanity Check ---')
print(f'Input row counts: {equation} = {sum_input_rows}')
print(f'Merged rows:      {merged_rows}')

if merged_rows == sum_input_rows:
    print('Sanity check: PASSED ✅  (merged rows equal sum of inputs)')
elif STOP_ON_SANITY_MISMATCH:
    msg = (f"Sanity check: FAILED ❌  (merged={merged_rows} vs sum={sum_input_rows}).\n"
           f"Common causes: blank lines parsed as rows in some CSVs, differing encodings, "
           f"rows dropped by parsing errors, extra BOM/header lines, or empty Excel sheets.")
    raise AssertionError(msg)
else:
    msg = (f"Sanity check: FAILED ❌  (merged={merged_rows} vs sum={sum_input_rows}).\n"
           f"Common causes: blank lines parsed as rows in some CSVs, differing encodings, "
           f"rows dropped by parsing errors, extra BOM/header lines, or empty Excel sheets.")
    print(msg)

# --- Clean data for Excel (remove illegal characters) ---
# Define a regex to match common illegal XML characters, including control characters
# The problematic character \x0b (vertical tab) is covered by this.
ILLEGAL_CHAR_RE = re.compile(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]')

for col in merged.select_dtypes(include=['object']).columns:
    merged[col] = merged[col].astype(str).apply(lambda x: ILLEGAL_CHAR_RE.sub('', x))

# --- Save to Drive ---
os.makedirs(OUTPUT_DRIVE_DIR, exist_ok=True)
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

if OUTPUT_FORMAT.lower() == 'xlsx':
    out_path = os.path.join(OUTPUT_DRIVE_DIR, f'{OUTPUT_BASENAME}_{timestamp}.xlsx')
    # Use index=False to match your CSV behavior
    with pd.ExcelWriter(out_path, engine='openpyxl') as writer:
        merged.to_excel(writer, index=False, sheet_name='merged')
else:
    out_path = os.path.join(OUTPUT_DRIVE_DIR, f'{OUTPUT_BASENAME}_{timestamp}.csv')
    # Keep your semicolon separator and utf-8-sig BOM for Excel-friendliness
    merged.to_csv(out_path, index=False, encoding='utf-8-sig', sep=';')

print('\nOutput saved to:')
print(out_path)

# Quick preview
display(merged.head(10))

Mounted at /content/drive
--- Merge Summary ---
Files merged: 3
- ne.xlsx: 3123 rows, 17 cols
- sa.xlsx: 66 rows, 17 cols
- sarl.xlsx: 8507 rows, 17 cols

--- Sanity Check ---
Input row counts: 3123 + 66 + 8507 = 11696
Merged rows:      11696
Sanity check: PASSED ✅  (merged rows equal sum of inputs)

Output saved to:
/content/drive/MyDrive/merged/merged_2025_herewego_20260512_100821.xlsx


,Statut,Annonce de statut,Jour,Date de création,Collaborateur,No collaborateur,UO,Prestation,No prestation,N° du client,Client,Début,Fin,Durée,Motif d'intervention,Manière d'enregistrement,Facturation
0,OK,nan,01.04.2026,01.04.2026 08:04:43,Mojsilovic Tamara,2014,Home Assistance (Neuchatel) SA,Temps de déplacement soins,61010,NaN,,01.04.2026 07:51,01.04.2026 08:02,11,nan,asebis smart,Non validé
1,OK,nan,01.04.2026,01.04.2026 08:30:24,Thévoz Martine,2026,Home Assistance (Neuchatel) SA,Temps de déplacement soins,61010,NaN,,01.04.2026 07:44,01.04.2026 08:20,36,nan,asebis smart,Non validé
2,OK,nan,01.04.2026,01.04.2026 10:43:15,Sihyürek Mine,1209,Home Assistance (Neuchatel) SA,Temps de déplacement soins,61010,NaN,,01.04.2026 10:08,01.04.2026 10:43,35,nan,asebis smart,Non validé
3,OK,nan,01.04.2026,01.04.2026 07:51:43,Mojsilovic Tamara,2014,Home Assistance (Neuchatel) SA,OPAS-B Examens et traitements,11100,103666.0,OVAREZ KUCUK Marie-Claude,01.04.2026 07:26,01.04.2026 07:51,25,Maladie,asebis smart,Non validé
4,OK,nan,01.04.2026,01.04.2026 17:40:41,Soler Fanny,2027,Home Assistance (Neuchatel) SA,Temps de déplacement soins,61010,NaN,,01.04.2026 17:18,01.04.2026 17:37,19,nan,asebis smart,Non validé
5,OK,nan,01.04.2026,01.04.2026 10:15:55,Thévoz Martine,2026,Home Assistance (Neuchatel) SA,Temps de déplacement soins,61010,NaN,,01.04.2026 09:53,01.04.2026 10:15,22,nan,asebis smart,Non validé
6,OK,nan,01.04.2026,01.04.2026 20:37:40,Mojsilovic Tamara,2014,Home Assistance (Neuchatel) SA,OPAS-C Soins de base,11200,104187.0,SINTZ Mireille,01.04.2026 20:25,01.04.2026 20:44,19,Maladie,asebis smart,Non validé
7,Ouvert,nan,01.04.2026,01.04.2026 21:24:48,Sutterlet Pascal,2023,Home Assistance (Neuchatel) SA,OPAS-C Soins de base,11200,103963.0,SUTTERLET Maryline,01.04.2026 19:00,01.04.2026 19:28,28,Maladie,asebis smart,Non validé
8,OK,nan,01.04.2026,01.04.2026 08:08:46,Incerti Mélanie,2004,Home Assistance (Neuchatel) SA,Temps de déplacement soins,61010,NaN,,01.04.2026 07:42,01.04.2026 08:07,25,nan,asebis smart,Non validé
9,OK,nan,01.04.2026,01.04.2026 18:52:46,Sihyürek Mine,1209,Home Assistance (Neuchatel) SA,OPAS-B Examens et traitements,11100,202863.0,LEUBA Erica,01.04.2026 18:25,01.04.2026 18:47,22,nan,asebis smart,Non validé
